# 2. Model Training - YOLOv11 & U-Net for Lunar Hazard Detection

This notebook covers training both YOLOv11 for boulder detection and U-Net for landslide segmentation on Chandrayaan imagery.

## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from ultralytics import YOLO
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## Prepare Training Data

In [ ]:
# Define data paths
data_path = Path('../data')
raw_path = data_path / 'raw'
processed_path = data_path / 'processed'
masks_path = data_path / 'masks'
models_path = Path('../models')

# Create models directory if it doesn't exist
models_path.mkdir(exist_ok=True)

print(f"Data paths configured:")
print(f"  Raw: {raw_path}")
print(f"  Processed: {processed_path}")
print(f"  Masks: {masks_path}")
print(f"  Models: {models_path}")

In [ ]:
class LunarImageDataset(Dataset):
    """Custom dataset for lunar hazard detection"""
    
    def __init__(self, image_dir, mask_dir=None, transform=None):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir) if mask_dir else None
        self.transform = transform
        self.image_files = sorted(self.image_dir.glob('*.tif')) + sorted(self.image_dir.glob('*.png'))
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        image = cv2.resize(image, (512, 512))
        
        if self.transform:
            image = self.transform(image)
        
        if self.mask_dir:
            mask_path = self.mask_dir / (img_path.stem + '_mask.png')
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) if mask_path.exists() else np.zeros_like(image)
            return torch.from_numpy(image).float().unsqueeze(0), torch.from_numpy(mask).float().unsqueeze(0)
        
        return torch.from_numpy(image).float().unsqueeze(0)

print("Dataset class defined")

## Train YOLOv11 for Boulder Detection

In [ ]:
# Initialize YOLOv11 model
# Note: Requires a properly formatted dataset in YOLO format

def train_yolo_model(epochs=50, batch_size=16):
    """
    Train YOLOv11 model for boulder detection
    
    Args:
        epochs: Number of training epochs
        batch_size: Batch size for training
    """
    # Load a pretrained YOLOv11 model
    model = YOLO('yolov11n.pt')  # nano model for faster training
    
    # Train the model
    # results = model.train(
    #     data='dataset.yaml',  # Path to YOLO dataset config
    #     epochs=epochs,
    #     imgsz=640,
    #     batch=batch_size,
    #     device=0,  # GPU device
    #     patience=10,
    #     save=True,
    #     verbose=True
    # )
    
    # Save trained model
    # model.save(str(models_path / 'boulder_yolo.pt'))
    print("YOLO training function defined")
    print("Note: Uncomment and configure with actual dataset to run training")

train_yolo_model()

## Build and Train U-Net for Landslide Segmentation

In [ ]:
class UNet(nn.Module):
    """U-Net architecture for semantic segmentation"""
    
    def __init__(self, in_channels=1, out_channels=1):
        super(UNet, self).__init__()
        
        # Encoder
        self.enc1 = self._conv_block(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = self._conv_block(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = self._conv_block(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = self._conv_block(256, 512)
        self.pool4 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = self._conv_block(512, 1024)
        
        # Decoder
        self.upconv4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = self._conv_block(1024, 512)
        self.upconv3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = self._conv_block(512, 256)
        self.upconv2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = self._conv_block(256, 128)
        self.upconv1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = self._conv_block(128, 64)
        
        # Final layer
        self.final = nn.Conv2d(64, out_channels, 1)
    
    def _conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder
        enc1 = self.enc1(x)
        x = self.pool1(enc1)
        enc2 = self.enc2(x)
        x = self.pool2(enc2)
        enc3 = self.enc3(x)
        x = self.pool3(enc3)
        enc4 = self.enc4(x)
        x = self.pool4(enc4)
        
        # Bottleneck
        x = self.bottleneck(x)
        
        # Decoder
        x = self.upconv4(x)
        x = torch.cat([x, enc4], dim=1)
        x = self.dec4(x)
        x = self.upconv3(x)
        x = torch.cat([x, enc3], dim=1)
        x = self.dec3(x)
        x = self.upconv2(x)
        x = torch.cat([x, enc2], dim=1)
        x = self.dec2(x)
        x = self.upconv1(x)
        x = torch.cat([x, enc1], dim=1)
        x = self.dec1(x)
        
        x = self.final(x)
        return x

print("U-Net architecture defined")

In [ ]:
def train_unet_model(epochs=30, batch_size=8, learning_rate=1e-4):
    """
    Train U-Net model for landslide segmentation
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Initialize model
    model = UNet(in_channels=1, out_channels=1).to(device)
    
    # Loss function and optimizer
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    print(f"Model initialized on {device}")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Training loop (placeholder)
    print("\nTraining configuration:")
    print(f"  Epochs: {epochs}")
    print(f"  Batch Size: {batch_size}")
    print(f"  Learning Rate: {learning_rate}")
    print("\nNote: Actual training requires prepared dataset")
    
    return model

# Initialize model
unet_model = train_unet_model()

## Model Evaluation and Metrics

In [ ]:
def calculate_metrics(y_true, y_pred):
    """
    Calculate precision, recall, F1-score, and IoU
    """
    # Flatten arrays
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    
    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(y_true_flat, y_pred_flat, average='binary')
    
    # Calculate IoU
    intersection = np.logical_and(y_true_flat, y_pred_flat).sum()
    union = np.logical_or(y_true_flat, y_pred_flat).sum()
    iou = intersection / union if union > 0 else 0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'iou': iou
    }

print("Metrics calculation function defined")

## Summary

This notebook provides:
1. **Data Preparation**: Custom dataset class for loading lunar imagery
2. **YOLOv11 Training**: Framework for boulder detection model training
3. **U-Net Architecture**: Complete U-Net implementation for segmentation
4. **Training Pipeline**: Setup for model training with proper configuration
5. **Metrics Evaluation**: Functions for computing precision, recall, F1, and IoU

To use this notebook:
1. Prepare your dataset in the appropriate format
2. Configure paths in the "Prepare Training Data" section
3. Uncomment and run the training functions
4. Monitor training metrics and save best models